In [2]:
import os
import cv2
import random

# =========================
# USER INPUT PATHS
# =========================

source_path = r"/mnt/lustre/lalit47/us_data/bench/ultrasound-nerve-segmentation"
target_path = r"/mnt/lustre/lalit47/down/seg/Neck Nerve"

# Source folders
train_source_folder = os.path.join(source_path, "train")
test_source_folder = os.path.join(source_path, "test")

# Target folders
train_images_out = os.path.join(target_path, "train", "images")
train_masks_out = os.path.join(target_path, "train", "masks")

val_images_out = os.path.join(target_path, "val", "images")
val_masks_out = os.path.join(target_path, "val", "masks")

test_images_out = os.path.join(target_path, "test", "images")

# Create target directories
os.makedirs(train_images_out, exist_ok=True)
os.makedirs(train_masks_out, exist_ok=True)

os.makedirs(val_images_out, exist_ok=True)
os.makedirs(val_masks_out, exist_ok=True)

os.makedirs(test_images_out, exist_ok=True)


def process_and_save(src, dst, is_mask=False):
    """
    Read image in grayscale,
    resize to 256x256,
    save as PNG.

    For masks, nearest-neighbor interpolation is used
    to preserve label values.
    """

    im = cv2.imread(src, cv2.IMREAD_GRAYSCALE)

    if im is None:
        print(f"Failed to read: {src}")
        return False

    if is_mask:
        im = cv2.resize(im, (256, 256), interpolation=cv2.INTER_NEAREST)
    else:
        im = cv2.resize(im, (256, 256), interpolation=cv2.INTER_LINEAR)

    cv2.imwrite(dst, im)
    return True


def get_train_image_names(train_folder):
    """
    Get image names from train folder.
    Excludes mask files ending with '_mask.tif'.
    """

    image_names = []

    for file in os.listdir(train_folder):
        file_lower = file.lower()

        if file_lower.endswith(".tif") and not file_lower.endswith("_mask.tif"):
            name = os.path.splitext(file)[0]
            image_names.append(name)

    image_names.sort()
    return image_names


def split_train_val(image_names, train_ratio=0.9, seed=42):
    """
    Split image names into train and validation sets.
    """

    random.seed(seed)
    random.shuffle(image_names)

    split_idx = int(len(image_names) * train_ratio)

    train_names = image_names[:split_idx]
    val_names = image_names[split_idx:]

    return train_names, val_names


def copy_train_val(names, img_output, mask_output):
    """
    Process train or val images and masks.
    Image format: name.tif
    Mask format: name_mask.tif
    """

    for name in names:
        image_src = os.path.join(train_source_folder, name + ".tif")
        mask_src = os.path.join(train_source_folder, name + "_mask.tif")

        image_dst = os.path.join(img_output, name + ".png")
        mask_dst = os.path.join(mask_output, name + ".png")

        if os.path.exists(image_src):
            process_and_save(image_src, image_dst, is_mask=False)
        else:
            print(f"Image Missing: {image_src}")

        if os.path.exists(mask_src):
            process_and_save(mask_src, mask_dst, is_mask=True)
        else:
            print(f"Mask Missing: {mask_src}")


def copy_test_images(test_folder, test_output):
    """
    Process test images only.
    Test folder has no masks.
    """

    test_files = []

    for file in os.listdir(test_folder):
        file_lower = file.lower()

        if file_lower.endswith(".tif"):
            test_files.append(file)

    test_files.sort()

    for file in test_files:
        name = os.path.splitext(file)[0]

        image_src = os.path.join(test_folder, file)
        image_dst = os.path.join(test_output, name + ".png")

        if os.path.exists(image_src):
            process_and_save(image_src, image_dst, is_mask=False)
        else:
            print(f"Test Image Missing: {image_src}")


# =========================
# Main processing
# =========================

all_train_names = get_train_image_names(train_source_folder)

train_names, val_names = split_train_val(
    all_train_names,
    train_ratio=0.9,
    seed=42
)

print(f"Total labeled images: {len(all_train_names)}")
print(f"Train images: {len(train_names)}")
print(f"Validation images: {len(val_names)}")

copy_train_val(train_names, train_images_out, train_masks_out)
copy_train_val(val_names, val_images_out, val_masks_out)

copy_test_images(test_source_folder, test_images_out)

print("\nDataset preprocessing completed successfully!")

Total labeled images: 5635
Train images: 5071
Validation images: 564

Dataset preprocessing completed successfully!
